# Модуль 14.5 — MCP: универсальный язык tool-сервера, своими руками

Этот ноутбук — практика к лекции «MCP: универсальный язык tool-сервера» (модуль 14.5 на сайте курса). Главный термин лекции — «MCP-сервер — это контракт, вынесенный в сервис»: все действия, данные и процедуры одного сервиса, отданные наружу по единому протоколу. Здесь вы выносите его своими руками.

Сквозной герой прежний — лесоруб из 11.5. Но чиним мы уже не tools и не skills: хороший контракт `move` / `gather` / `deposit` / `get_map` достаётся готовым. Симптом теперь другой: контракт заперт внутри одного проекта — под каждого нового клиента приходится писать обвязку заново, а чужие приложения (Claude Desktop, Cursor) не подключить вовсе. Лекарство — MCP-сервер: описать инструменты один раз и дать любому клиенту найти их по протоколу.

**Что вы получите на выходе:**

- свой MCP-сервер лесоруба на FastMCP: tools `move` / `gather` / `deposit` плюс resource `woodcutter://map` — файл, который поднимает любой MCP-клиент;
- discovery руками: `tools/list`, `resources/list`, `tools/call` через fastmcp-клиент — in-process и по stdio;
- эксперимент «убери инструмент»: клиент видит живой каталог, а не ваш код;
- клиент №1 — smolagents-агент через `ToolCollection.from_mcp` рубит мок-лес по MCP (опционально, нужна локальная модель);
- клиент №2 — MCP Inspector и Claude Code: те же инструменты видны из клиента, который вы не писали;
- MCP-обёртку живого игрового API Cognopolis — агент управляет настоящим жителем через ваш сервер (опционально, нужен под-токен).

**Карта ноутбука:**

- **Блок 1 (ядро, keyless)** — симптом M×N: обвязка на каждого клиента.
- **Блок 2 (ядро, keyless)** — свой MCP-сервер: файл, три примитива, discovery и вызовы in-process.
- **Блок 3 (ядро, keyless)** — транспорт stdio и «убери инструмент»: discovery вживую.
- **Блок 4 (опционально, нужен локальный сервер модели)** — клиент №1: smolagents через MCP.
- **Блок 5 (опционально, нужны под-токен и локальный сервер)** — живая игра: MCP-обёртка Cognopolis.
- **Блок 6** — клиент №2, который вы не писали: MCP Inspector и Claude Code.
- **Задачи** — второй resource, карта-как-tool, новый узкий tool живого сервера, мостик в Claude Desktop.

Главное про запуск: ядро исполняется **целиком и без единого ключа** — `Run all`, без правок, без сети к игре. Интернет нужен только в начале — поставить библиотеки, а в Colab/Kaggle ещё скачать серверные файлы из репозитория (ячейка «Файлы урока» сделает это сама). Блоки 4–5 без локального сервера модели и под-токена делают мягкий пропуск (soft-skip) — keyless-прогон остаётся зелёным. Артефакт модуля — файл `mcp-server/woodcutter_server.py` (а для смелых — `mcp-server/cognopolis_mcp_server.py`), который видят два разных клиента.

## Подготовка окружения

Ставим три библиотеки: `fastmcp` (сервер и клиент MCP на Python — тот же `@mcp.tool` / `@mcp.resource`, что в лекции), `smolagents[mcp]` (движок из Модуля 10; extra `[mcp]` дотягивает адаптер для `ToolCollection.from_mcp`) и `openai` (клиент OpenAI-совместимых серверов — нужен `OpenAIServerModel` в Блоках 4–5).

Пометка про Python: `fastmcp` требует **Python 3.10+**. В Colab и Kaggle это уже так; локально проверьте `python --version` и при нужде создайте venv поновее (например, `uv venv --python 3.11`). Ячейка ниже проверит версию сама.

Честная пометка про сеть: **для установки нужен интернет**. В Colab он есть всегда; в Kaggle включите `Notebook options → Internet → On`. После установки и ячейки «Файлы урока» всё ядро работает без сети. API-ключи не нужны нигде: Блокам 4–5 достаточно локального сервера модели и под-токена жителя.

In [ ]:
import sys

assert sys.version_info >= (3, 10), (
    f"fastmcp требует Python 3.10+, у вас {sys.version_info.major}.{sys.version_info.minor}. "
    "В Colab/Kaggle это уже так; локально создайте venv поновее, например: uv venv --python 3.11"
)

%pip install -q fastmcp "smolagents[mcp]" openai

import fastmcp
import openai
import smolagents

print("Python", sys.version.split()[0],
      "| fastmcp", fastmcp.__version__,
      "| smolagents", smolagents.__version__,
      "| openai", openai.__version__)

## Файлы урока: папка `mcp-server/`

Серверы этого модуля — не ячейки, а настоящие файлы: они живут в папке `mcp-server/` рядом с ноутбуком (в клоне репозитория она уже есть). Так и должно быть — MCP-сервер живёт отдельным процессом, который клиент запускает сам, поэтому и артефакт домашки — файлы, а не код в ноутбуке.

Ячейка ниже наводит порядок в любом окружении. Если папки с файлами нет — так бывает в Colab и Kaggle, куда ноутбук приезжает одним файлом, — она скачает канонические серверы raw-ссылками из репозитория курса. Затем запомнит абсолютный путь в `SERVER_DIR` (через него пути получают все ячейки с `Client(...)` и `StdioServerParameters`) и добавит папку в `sys.path`, чтобы in-process ячейки могли делать обычный `import woodcutter_server`.


In [ ]:
import sys
from pathlib import Path
from urllib.request import urlretrieve

SERVER_DIR = Path("mcp-server").resolve()   # тут живут файлы серверов
SERVER_DIR.mkdir(exist_ok=True)

RAW_BASE = ("https://raw.githubusercontent.com/ITrubnikov/Train_of_Thought-homework/"
            "main/notebooks/module-14-5-mcp/mcp-server")

for fname in ("woodcutter_server.py", "cognopolis_mcp_server.py"):
    target = SERVER_DIR / fname
    if target.exists():
        print(f"{fname}: на месте")
    else:   # Colab/Kaggle: ноутбук открыт без папки - скачиваем канонический файл
        urlretrieve(f"{RAW_BASE}/{fname}", target)
        print(f"{fname}: скачан из репозитория")

if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))   # для import woodcutter_server в in-process ячейках

print("SERVER_DIR =", SERVER_DIR)


## Блок 1 (ядро, keyless). Симптом: контракт заперт внутри проекта

В 11.5 вы спроектировали контракт лесоруба, в 11.7 — объявляли его руками под каждый движок: `@tool` для smolagents, `input_schema` для голого Anthropic API. Пока клиент один — терпимо. Но клиентов становится больше, и каждый требует свою обвязку:

```text
лесоруб (те же move/gather/deposit)
├── smolagents-агент     → обвязка №1: @tool + agent.run()
├── Claude Desktop       → обвязка №2: ??? нет способа импортировать ваши функции
├── агент в Cursor       → обвязка №3: ??? то же самое
└── голый Anthropic API  → обвязка №4: input_schema руками
```

Два клиента из четырёх не подключить вовсе: Claude Desktop и Cursor — чужие приложения, в которые ваш Python-модуль не импортируешь. Добавится второй сервис (не лесоруб, а, скажем, ваша база данных) — и всё умножается заново. Это классическая проблема «M клиентов × N сервисов».

Заметьте, чего здесь не хватает. Контракт-то один и тот же: `move(direction)` с направлениями, обучающие ошибки `{code, message}`, кулдаун. Не хватает **общего языка**, на котором сервис объявил бы контракт один раз, а любой клиент прочитал. Этот язык — MCP, и главный термин лекции звучит так: **MCP-сервер — это контракт, вынесенный в сервис**. Не новый способ вызывать функции вместо tool calling, а способ раздать тот же контракт всем клиентам сразу.

In [ ]:
clients = ["smolagents-агент", "Claude Desktop", "агент в Cursor", "голый Anthropic API"]
services = ["лесоруб", "база данных", "GitHub"]

print(f"Без MCP: {len(clients)} клиента x {len(services)} сервиса "
      f"= {len(clients) * len(services)} обвязок, каждую писать и чинить руками")
print(f"С MCP:   {len(clients)} + {len(services)} "
      f"= {len(clients) + len(services)} реализаций протокола: сервис описывает себя один раз,")
print("         клиент один раз учит MCP - и любая пара совместима без клей-кода")

## Блок 2 (ядро, keyless). Свой MCP-сервер: файл и три примитива

Чиним. Писать протокол руками не нужно — библиотека FastMCP собирает MCP-сервер из обычных Python-функций: `@mcp.tool` делает из функции инструмент (схему аргументов соберёт из type hints, описание — из docstring, ровно как smolagents в 11.7), `@mcp.resource("uri")` объявляет данные на чтение, `mcp.run()` поднимает транспорт.

Сервер ниже — лесоруб из лекции. Мок-лес тот же, что в 11.5 и 13.6: сетка 5×5, дерево на `(1, 2)` и `(3, 1)`, камень на `(2, 4)`, рюкзак на 5, склад на `(0, 0)`; контракт tools `move` / `gather` / `deposit` с конвертом `{result, cooldown}` и обучающими ошибками `{error: {code, message}}`. Единственное отличие от лекции — `COOLDOWN = 0.3` вместо 1.0, только ради времени `Run all`; правила те же.

Важно: это не ячейка с функциями, а **файл** — `mcp-server/woodcutter_server.py`, он уже лежит в папке (ячейка «Файлы урока» позаботилась об этом и в Colab). MCP-сервер живёт отдельным процессом, который клиент запускает сам, — потому и файл. Он и есть артефакт модуля: откройте его целиком в папке, а ячейка ниже напечатает ключевые фрагменты — сервер, tool `move` и resource-карту.

In [ ]:
server_path = SERVER_DIR / "woodcutter_server.py"
server_src = server_path.read_text(encoding="utf-8")
print(f"mcp-server/{server_path.name}: {len(server_src.splitlines())} строк. Ключевые фрагменты:")
print()
print(server_src[server_src.index("mcp = FastMCP("):server_src.index("@mcp.tool\ndef gather")].rstrip())
print("...")
print(server_src[server_src.index("@mcp.resource"):server_src.index("if __name__")].rstrip())
print()
print("Целиком - в файле: мок-лес Forest, tools move/gather/deposit с обучающими")
print("ошибками {error: {code, message}} и resource-карта. Прочитайте его весь.")


### Discovery: клиент спрашивает, сервер отвечает

Сервер есть — подключим клиента. Первый клиент — из самой fastmcp: `Client` умеет подключаться и к объекту сервера в этом же процессе (in-process — удобно для отладки), и к файлу по stdio (это Блок 3). Клиент асинхронный, но в ноутбуках работает top-level `await` — IPython сам крутит event loop.

Первое, что делает любой MCP-клиент после подключения, — **discovery**: спрашивает сервер, что тот умеет. Не читает ваш код — именно спрашивает по протоколу: `tools/list`, `resources/list`, `prompts/list`. Посмотрите на схему `move` в ответе: её никто не писал руками — FastMCP собрал её из сигнатуры и docstring.

In [ ]:
import importlib
import json

from fastmcp import Client

import woodcutter_server as ws

ws = importlib.reload(ws)   # перечитать файл, если вы правили его и перезапускаете ячейки

async with Client(ws.mcp) as client:            # in-process: клиент и сервер в одном процессе
    tools = await client.list_tools()           # tools/list
    resources = await client.list_resources()   # resources/list
    prompts = await client.list_prompts()       # prompts/list

print("tools/list:")
for t in sorted(tools, key=lambda t: t.name):
    print(f"  {t.name:<8} - {(t.description or 'без описания').splitlines()[0]}")

move_tool = next(t for t in tools if t.name == "move")
print()
print("Схема move, собранная из type hints и docstring:")
print(json.dumps(move_tool.inputSchema, ensure_ascii=False, indent=2))

print()
print("resources/list:", [str(r.uri) for r in resources])
print("prompts/list:  ", [p.name for p in prompts] or "пусто - наш сервер prompts не объявляет")

assert sorted(t.name for t in tools) == ["deposit", "gather", "move"]
assert [str(r.uri) for r in resources] == ["woodcutter://map"]

### tools/call: цепочка вызовов с обучающими ошибками

Discovery показал каталог — теперь вызовы. `call_tool(имя, аргументы)` — это `tools/call` из протокола; у результата есть `.data` — уже распарсенный ответ инструмента. Прогоним цепочку, в которой агент ошибается всеми способами из 11.5, — и убедимся, что обучающие ошибки доехали через протокол без потерь: тот же `{error: {code, message}}`, что вы проектировали, только теперь его получает любой MCP-клиент.

Два места, на которые стоит смотреть. `on_cooldown`: сервер сам говорит, сколько ждать, — ритм держит сервис, а не вежливость клиента. И `unknown_direction`: в этом сервере `direction` объявлен как `str`, так что мусорное направление долетает до функции, и её спасает обучающая ошибка. В живом сервере Блока 5 то же место закрыто раньше — `Literal` кладёт enum прямо в схему, и клиент отсечёт мусор ещё до вызова. Два уровня защиты из 11.5, теперь по разные стороны протокола.

In [ ]:
import time

ws.forest = ws.Forest()   # свежий мир: tools сервера смотрят на woodcutter_server.forest


def show(step, res):
    err = res.get("error")
    verdict = f"error {err['code']}" if err else f"ok {res['result']}"
    print(f"  {step:<26} -> {verdict}")


async with Client(ws.mcp) as client:
    r1 = (await client.call_tool("gather", {})).data
    show("gather() на складе", r1)                    # no_resource_here: узла тут нет
    r2 = (await client.call_tool("move", {"direction": "west"})).data
    show('move("west") с края', r2)                   # at_map_edge: дальше леса нет
    r3 = (await client.call_tool("move", {"direction": "up"})).data
    show('move("up")', r3)                            # unknown_direction: схема-то str
    r4 = (await client.call_tool("move", {"direction": "east"})).data
    show('move("east")', r4)                          # ok: (1, 0), пошёл кулдаун
    r5 = (await client.call_tool("move", {"direction": "south"})).data
    show('move("south") сразу же', r5)                # on_cooldown: сервер держит ритм
    time.sleep(0.4)
    r6 = (await client.call_tool("move", {"direction": "south"})).data
    show('move("south") подождав', r6)                # ok: (1, 1)
    time.sleep(0.4)
    r7 = (await client.call_tool("move", {"direction": "south"})).data
    show('move("south")', r7)                         # ok: (1, 2) - тут дерево
    time.sleep(0.4)
    r8 = (await client.call_tool("gather", {"resource": "wood"})).data
    show('gather("wood")', r8)                        # ok: wood в рюкзаке

for res, code in [(r1, "no_resource_here"), (r2, "at_map_edge"),
                  (r3, "unknown_direction"), (r5, "on_cooldown")]:
    assert res["error"]["code"] == code, res
assert r8["result"]["backpack"] == {"wood": 1}
print()
print("Все ошибки доехали через протокол как {error: {code, message}} - контракт 11.5 цел.")

### resources/read и три примитива

Карта в этом сервере — не tool, а **resource** с URI `woodcutter://map`: лесоруб её только читает. Прочитаем — и сверим с миром: после цепочки выше лесоруб стоит на `(1, 2)` с деревом в рюкзаке.

Теперь разложим по полочкам все **три примитива** MCP-сервера — три вида того, что сервер отдаёт клиенту:

- **tool** — действие, меняет мир: `move`, `gather`, `deposit`. Аналог `POST`. Именно поэтому у него кулдаун и обучающие ошибки — всё, за что вы боролись в 11.5.
- **resource** — данные только на чтение, со своим URI: `woodcutter://map`. Аналог `GET`: клиент подтягивает содержимое в контекст, не «вызывая действие».
- **prompt** — готовый шаблон-процедура от сервера, родственник skill из 13.6, только приходит со стороны сервиса, а не заучивается агентом. Наш сервер их не объявляет — потому `prompts/list` и был пуст.

Микропроверка из лекции: `deposit` меняет склад — tool; карта только читается — resource. Спутать их — как перепутать `GET` и `POST`: ресурс, оформленный как tool, заставит модель «вызывать действие» там, где нужно просто прочитать контекст. В Задаче 2 вы устроите эту путаницу нарочно и посмотрите на последствия.

In [ ]:
async with Client(ws.mcp) as client:
    contents = await client.read_resource("woodcutter://map")   # resources/read по URI

map_data = json.loads(contents[0].text)   # содержимое ресурса - текст; наш сервер отдаёт JSON
print("woodcutter://map:", json.dumps(map_data, ensure_ascii=False))

assert map_data["pos"] == [1, 2], "карта живая: лесоруб там, куда дошёл в прошлой ячейке"
assert ws.forest.backpack == {"wood": 1}, "и рюкзак не потерялся - мир один"
print()
print("Ресурс читает живое состояние: pos на карте совпал с миром после наших move.")

## Блок 3 (ядро, keyless). Транспорт: stdio-подпроцесс

До сих пор клиент и сервер жили в одном процессе — удобно для отладки, но нечестно: настоящий клиент ваш Python-объект не импортирует. Честный локальный способ — транспорт **stdio**: клиент сам запускает сервер подпроцессом (`python woodcutter_server.py`) и обменивается с ним JSON-RPC-сообщениями через stdin/stdout. Ноль сети, ноль портов — это дефолт `mcp.run()`. Второй транспорт — Streamable HTTP — для сервера, который живёт в сети и обслуживает многих; контракт инструментов от смены транспорта не меняется вообще.

`Client` из fastmcp понимает путь к файлу: `Client(str(SERVER_DIR / "woodcutter_server.py"))` значит «подними подпроцесс `python woodcutter_server.py` и говори с ним по stdio» — путь абсолютный, из ячейки «Файлы урока». И заметьте: каталог инструментов тот же, что in-process, а вот мир — свежий: подпроцесс исполнил файл заново, `forest` в нём свой.

In [ ]:
async with Client(str(SERVER_DIR / "woodcutter_server.py")) as client:   # stdio-подпроцесс
    stdio_tools = sorted(t.name for t in await client.list_tools())
    stdio_map = json.loads((await client.read_resource("woodcutter://map"))[0].text)

print("tools/list по stdio:", stdio_tools)
print("Карта из подпроцесса:", json.dumps(stdio_map, ensure_ascii=False))

assert stdio_tools == ["deposit", "gather", "move"], "каталог тот же, что in-process"
assert stdio_map["pos"] == [0, 0], "а мир свежий: подпроцесс поднял свой Forest с нуля"
print()
print("Тот же каталог, что in-process, - но лесоруб снова на (0, 0): это другой процесс.")

### Эксперимент «убери инструмент»: клиент видит каталог, а не код

Шаг 5 домашки лекции. Клиент не читает ваши исходники — он спрашивает `tools/list` вживую. Проверим следствие: уберём из сервера `deposit` и посмотрим, что увидит клиент. В жизни вы бы отредактировали сам `woodcutter_server.py`; здесь, чтобы канонический файл остался цел для остальных ячеек, ячейка ниже готовит урезанную копию кодом: читает `woodcutter_server.py`, вырезает функцию `deposit` — от её `@mcp.tool` до следующего примитива — и записывает результат в `mcp-server/woodcutter_server_v2.py`. Этот код и есть инструкция «уберите инструмент», только выполненная автоматически.

In [ ]:
src = (SERVER_DIR / "woodcutter_server.py").read_text(encoding="utf-8")

start = src.index("@mcp.tool\ndef deposit")   # начало deposit вместе с декоратором
end = src.index("@mcp.resource")              # следующий примитив после него
v2_src = (
    "# Лесоруб v2 — эксперимент «убери инструмент» из домашки лекции 14.5: тот же сервер,\n"
    "# но deposit вырезан. Файл сгенерирован ноутбуком из woodcutter_server.py.\n"
    + src[:start] + src[end:]
).replace("python woodcutter_server.py", "python woodcutter_server_v2.py")   # шапка запуска

assert "def deposit" not in v2_src and "def gather" in v2_src

(SERVER_DIR / "woodcutter_server_v2.py").write_text(v2_src, encoding="utf-8")
print("Записан mcp-server/woodcutter_server_v2.py - тот же сервер, минус функция deposit.")


In [ ]:
async with Client(str(SERVER_DIR / "woodcutter_server_v2.py")) as client:
    v2_tools = sorted(t.name for t in await client.list_tools())

print("tools/list у v2:", v2_tools)
assert v2_tools == ["gather", "move"], "deposit исчез из каталога вместе с функцией"
print()
print("Никаких правок на стороне клиента: убрали функцию из сервера - действие пропало")
print("из tools/list при следующем же запросе. Мёртвых заглушек в живом каталоге не бывает.")

### Что летает по проводу: JSON-RPC 2.0

Всё это время клиент и сервер обменивались сообщениями **JSON-RPC 2.0** — слой протокола одинаков для обоих транспортов. Когда вы звали `client.list_tools()`, в stdin подпроцесса улетала строка вида:

```json
{"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
```

а из stdout приходил ответ с каталогом:

```json
{"jsonrpc": "2.0", "id": 1, "result": {"tools": [{"name": "move", "inputSchema": {"...": "..."}}]}}
```

`call_tool` — это метод `tools/call` с именем и аргументами, `read_resource` — `resources/read` с URI. FastMCP прячет эту кухню с обеих сторон: сервер собирает ответы из ваших функций, клиент превращает методы протокола в удобные вызовы. Но полезно один раз увидеть, что «универсальный язык» — это буквально несколько методов поверх JSON-RPC: именно поэтому клиент на любом языке может говорить с сервером на любом.

Ядро собрано: сервер-файл, три примитива, discovery, транспорт. Осталось главное доказательство — что к серверу подключаются клиенты, которых вы не писали с ним в паре. Их будет два: свой агент (Блок 4) и чужое приложение (Блок 6).

## Блок 4 (опционально, нужен локальный сервер модели). Клиент №1: smolagents через MCP

В Модуле 10 вы подключали к smolagents чужой MCP-сервер через `ToolCollection.from_mcp`. Теперь по ту сторону провода — ваш собственный: `StdioServerParameters` описывает команду запуска (`python mcp-server/woodcutter_server.py`), адаптер поднимает подпроцесс, делает discovery и превращает найденные tools в инструменты smolagents. Никакой обвязки: агент не знает, что внутри сервера мок-лес, — он видит стандартный каталог.

Модель — целиком на вашей машине, тем же приёмом, что в 13.6: любой OpenAI-совместимый сервер — LM Studio (`http://localhost:1234/v1`) или Ollama (`http://localhost:11434/v1`) с instruct-моделью, умеющей tool calling (например, `qwen2.5:7b`). Если сервер живёт по другому адресу, задайте переменные окружения `LOCAL_API_BASE` и `LOCAL_MODEL_ID` — ячейка проверит их первыми. Блок **необязательный**: сервера нет — ячейки печатают причину и мягко пропускаются; в Colab/Kaggle локального сервера не бывает, там блок пропустится всегда — запускайте его на своей машине (вариант C из README).

Одна деталь, ради которой блок стоит запустить даже глазами: `ToolCollection.from_mcp` подтягивает **только tools** — resource `woodcutter://map` до агента не доедет. Для мок-леса выход простой: карта маленькая, положим её словами в instructions. В живом сервере Блока 5 та же проблема решена иначе — карта отдана и как resource (правильный примитив для чтения), и как tool `get_map` (чтобы агентные клиенты вроде `from_mcp` её видели). Это осознанный трейд-офф из лекции, а не ошибка дизайна; в Задаче 2 вы прочувствуете его руками.

In [ ]:
import os

import requests

CANDIDATES = [
    ("LM Studio", "http://localhost:1234/v1"),
    ("Ollama", "http://localhost:11434/v1"),
]
if os.environ.get("LOCAL_API_BASE"):
    CANDIDATES.insert(0, ("LOCAL_API_BASE", os.environ["LOCAL_API_BASE"]))

LOCAL_BASE = None    # адрес найденного сервера
LOCAL_MODEL = None   # выбранная модель

for server_name, base in CANDIDATES:
    try:
        r = requests.get(f"{base}/models", timeout=2)
        r.raise_for_status()
        ids = [item["id"] for item in r.json().get("data", [])]
    except Exception:
        continue
    chat_ids = [i for i in ids if "embed" not in i.lower()]   # embedding-модели не годятся
    if chat_ids:
        LOCAL_BASE = base
        LOCAL_MODEL = os.environ.get("LOCAL_MODEL_ID", chat_ids[0])
        print(f"{server_name}: сервер найден на {base} -> модель {LOCAL_MODEL}")
        break

if LOCAL_BASE is None:
    print("Локальный сервер не найден -> живые прогоны Блоков 4-5 пропустятся (для keyless это норма).")

In [ ]:
MOCK_INSTRUCTIONS = (
    "Ты - лесоруб в мок-лесу 5x5 (x и y от 0 до 4; north уменьшает y, south увеличивает, "
    "east увеличивает x, west уменьшает). Ты стоишь на (0, 0), там же склад. "
    "Дерево (wood) лежит на (1, 2) и (3, 1), камень (stone) - на (2, 4). Рюкзак вмещает 5. "
    "Действуй только инструментами. Ошибка инструмента - не провал, а подсказка: читай code "
    "и message; on_cooldown значит подожди долю секунды и повтори тот же вызов. "
    "deposit работает только на складе (0, 0)."
)

if LOCAL_BASE is None:
    print("Блок 4 пропущен: локальный сервер модели не найден (LM Studio :1234 / Ollama :11434).")
else:
    try:
        import sys

        from mcp import StdioServerParameters
        from smolagents import OpenAIServerModel, ToolCallingAgent, ToolCollection

        params = StdioServerParameters(
            command=sys.executable,
            args=[str(SERVER_DIR / "woodcutter_server.py")],
            env={**os.environ},
        )
        with ToolCollection.from_mcp(params, trust_remote_code=True) as tc:
            print("Инструменты, доехавшие до smolagents:", sorted(t.name for t in tc.tools))
            print("(resource woodcutter://map не доехал - from_mcp умеет только tools)")
            print()
            agent = ToolCallingAgent(
                tools=[*tc.tools],
                model=OpenAIServerModel(model_id=LOCAL_MODEL, api_base=LOCAL_BASE,
                                        api_key="local"),   # заглушка: локальный сервер ключ не проверяет
                max_steps=15,   # мок-задаче хватает ~10 действий; лимит держит ритм лекции и Run all
                instructions=MOCK_INSTRUCTIONS,
            )
            agent.run("Добудь два дерева (wood) и сдай их на склад через deposit.")

        calls = [t.name for step in agent.memory.steps
                 for t in (getattr(step, "tool_calls", None) or [])]
        print()
        print("Факт-чек по agent.memory.steps - цепочка вызовов:", calls)
        print(f"gather вызван {calls.count('gather')} раз, deposit - {calls.count('deposit')};")
        print("агент рубил мок-лес через ваш MCP-сервер, не увидев ни строчки его кода.")
    except Exception as e:
        print("Живой прогон не прошёл -> мягкий пропуск:", repr(e))

## Блок 5 (опционально, нужны под-токен и локальный сервер). Живая игра: MCP-обёртка Cognopolis

Мок своё отработал — оборачиваем **живой игровой API** [kindomklaster.com](https://kindomklaster.com). Сервер — файл `mcp-server/cognopolis_mcp_server.py`, он тоже уже в папке — та самая «обёртка над игровым API» из домашки лекции: контракт tools дословно лесоруб (`move` / `gather` / `get_character` / `get_map`), но за ними не мок, а настоящий житель поселения; конверт живого мира — `{result, cooldown, character}`. Три отличия от мок-сервера, все по урокам этого модуля:

- `move` объявлен через `Literal["north", "south", "east", "west"]` — в схему `tools/list` попадает честный enum, и клиент отсечёт мусорное направление ещё до вызова;
- карта отдана дважды: resource `cognopolis://map` (правильный примитив для чтения) и tool `get_map` (для агентных клиентов, которые умеют только tools) — тот самый трейд-офф из Блока 4;
- отдельного `deposit` нет: в живом мире шаг на дом `(0, 0)` сам сдаёт рюкзак на склад — в ответе `move` это поле `banked`.

Ячейка ниже печатает ключевые фрагменты файла — `move` с `Literal` и карту двумя примитивами; целиком читайте его в `mcp-server/`.

Под-токен жителя сервер берёт **только из переменной окружения** `COGNOPOLIS_TOKEN` — при stdio-запуске клиент сам передаёт её подпроцессу, в протокол и в код токен не попадает. Возьмите под-токен своего жителя в интерфейсе игры (экран «Жители») — или тестового аккаунта, если преподаватель дал доступ. Токена нет — живые ячейки блока мягко пропускаются.

### Прод или локальная игра

Обёртка ходит в тот мир, который назван в переменной `COGNOPOLIS_BASE_URL`; по умолчанию это живая игра `https://kindomklaster.com`, и менять ничего не нужно. Если у вас запущен **локальный движок игры** — задайте `COGNOPOLIS_BASE_URL=http://localhost:8000` (переменной окружения до запуска Jupyter или строкой в `mcp-server/.env`), и сервер целиком переключится на него. Контракт tools, клиенты и их конфиги при этом не меняются вовсе — тот же трюк «меняется провод, не контракт», что и с транспортом.

Одна гоча: **под-токен должен быть из того же мира**. У локальной игры свои жители и свои токены (копируются на её экране «Жители», как и на проде) — прод-токен против локального движка даст `invalid_token`, и наоборот. Переключая мир, меняйте и токен. Какой мир выбран сейчас, печатает ячейка токена ниже.

In [ ]:
live_path = SERVER_DIR / "cognopolis_mcp_server.py"
live_src = live_path.read_text(encoding="utf-8")
print(f"mcp-server/{live_path.name}: {len(live_src.splitlines())} строк. Ключевые фрагменты:")
print()
print(live_src[live_src.index("@mcp.tool\ndef move"):live_src.index("@mcp.tool\ndef gather")].rstrip())
print("...")
print(live_src[live_src.index("@mcp.tool\ndef get_map"):live_src.index("if __name__")].rstrip())
print()
print("Целиком - в файле: конверт живого мира {result, cooldown, character} и ошибки")
print("{error: {code, message}} собирает вспомогательная _request - прочитайте её тоже.")


### Под-токен: только env или getpass, никогда — текстом в ячейке

Правило то же, что во всех уроках с живой игрой: токен не должен попасть ни в код, ни в вывод, ни в git. Ячейка ниже смотрит в переменную окружения `COGNOPOLIS_TOKEN`; если её нет, а вы поставили `ASK_TOKEN = True`, — спросит токен через `getpass` (ввод не отображается и в ноутбуке не сохраняется). По умолчанию `ASK_TOKEN = False`, чтобы `Run all` не останавливался на вводе.

In [ ]:
import os

ASK_TOKEN = False   # True -> спросить под-токен через getpass (Run all остановится и будет ждать ввода)

COGNOPOLIS_TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")
if not COGNOPOLIS_TOKEN:            # третий источник - mcp-server/.env (шаблон .env-example)
    env_file = SERVER_DIR / ".env"
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            if line.strip().startswith("COGNOPOLIS_TOKEN="):
                COGNOPOLIS_TOKEN = line.split("=", 1)[1].strip().strip("'\"")
if not COGNOPOLIS_TOKEN and ASK_TOKEN:
    from getpass import getpass
    COGNOPOLIS_TOKEN = getpass("Под-токен жителя Cognopolis: ").strip()

WORLD = os.environ.get("COGNOPOLIS_BASE_URL", "https://kindomklaster.com")
print("Мир игры:", WORLD, "(переключение - COGNOPOLIS_BASE_URL, см. разбор выше)")
if COGNOPOLIS_TOKEN:
    os.environ["COGNOPOLIS_TOKEN"] = COGNOPOLIS_TOKEN   # отсюда его возьмут сервер и подпроцесс
    print("Токен получен (печатать не будем). Живые ячейки Блока 5 активны.")
else:
    print("Токена нет -> живые ячейки Блока 5 мягко пропускаются. Ядро от этого не страдает.")

### Прямая проверка: get_character и resource cognopolis://map

Прежде чем сажать за сервер агента, проверим обёртку руками — тем же fastmcp-клиентом, что в Блоке 2 (in-process: сервер импортируется в этот же процесс и читает токен из окружения). Смотрим три вещи: enum в схеме `move` (спасибо `Literal`), живого жителя из `get_character` и карту поселения из resource. Такт здесь длиннее мока и разный по действию: шаг ~1 с, добыча ~10 с, крафт — десятки секунд (точные числа мир отдаёт сам: `GET /stats` → `tact.seconds`, длительность рецепта — `GET /recipes` → `seconds`). Живой мир держит ритм строже мока, и ждать его теперь заметно дольше.

In [ ]:
if not COGNOPOLIS_TOKEN:
    print("Пропуск: под-токена нет - прямую проверку живого сервера не запускаем.")
else:
    try:
        import cognopolis_mcp_server
        importlib.reload(cognopolis_mcp_server)   # перечитать COGNOPOLIS_TOKEN, если ячейку перезапускали

        async with Client(cognopolis_mcp_server.mcp) as client:
            live_tools = await client.list_tools()
            print("tools/list:", sorted(t.name for t in live_tools))
            move_t = next(t for t in live_tools if t.name == "move")
            print("Схема move.direction:",
                  json.dumps(move_t.inputSchema["properties"]["direction"], ensure_ascii=False))
            char = (await client.call_tool("get_character", {})).data
            print("Житель:", char.get("name"), "| позиция:", (char.get("x"), char.get("y")),
                  "| hp:", char.get("hp"), "| склад:", char.get("stored"))
            mp = json.loads((await client.read_resource("cognopolis://map"))[0].text)
            print(f"resource cognopolis://map: тайлов {len(mp.get('tiles', []))}, "
                  f"живых врагов {len(mp.get('enemies', []))}")
        assert "north" in json.dumps(move_t.inputSchema), "Literal доехал до схемы enum-ом"
    except Exception as e:
        print("Живая проверка не прошла -> мягкий пропуск:", repr(e))

### Агент на живом сервере: «добудь дерево и вернись домой»

Теперь агент. Схема та же, что в Блоке 4, — `ToolCollection.from_mcp` плюс `StdioServerParameters`, — но сервер живой, и токен передаётся подпроцессу явно, через `env`. В instructions — два правила, без которых живой мир агента запутает: такт действия (`character_on_cooldown` — не провал) и авто-банк на `(0, 0)` — иначе агент будет искать несуществующий `deposit`. Про такт важно понимать, кто его ждёт: сам агент спать не умеет, у него нет такой тулы. Поэтому ждёт **обёртка** — в `cognopolis_mcp_server.py` действия ходят через `_act`, который досиживает такт по полю `cooldown` ответа. Это и есть правильный приём: длительность берётся из ответа сервера, а не зашивается числом, — иначе на десятисекундной добыче агент сожжёт все свои шаги на повторы.

Факт-чек — не по словам агента, а по миру: `stored.wood` до и после. Дельта может оказаться больше 1: если на складе поселения лежат инструменты (например, топор), добыча за один `gather` растёт — инструменты общие. Прогон занимает пару минут: живой кулдаун честно ждётся.

In [ ]:
LIVE_INSTRUCTIONS = (
    "Ты управляешь жителем поселения Cognopolis через инструменты MCP-сервера. "
    "Правила живого мира: после действия житель занят - шаг около секунды, добыча около "
    "десяти; обёртка досиживает такт за тебя, поэтому просто зови следующий вызов. Получив "
    "ошибку "
    "character_on_cooldown, просто повтори тот же вызов. Дом - клетка (0, 0): шаг на неё "
    "сам сдаёт рюкзак на склад (в ответе move это поле banked), отдельного deposit нет - "
    "не ищи его. Карта - инструмент get_map: тайл tree это дерево (wood), rock - камень (stone). "
    "Начни с get_character и get_map: найди ближайший к себе тайл tree и иди к нему - "
    "каждый move должен приближать к цели, не блуждай."
)

if not COGNOPOLIS_TOKEN:
    print("Пропуск: под-токена нет - живой агент не запускается.")
elif LOCAL_BASE is None:
    print("Пропуск: локального сервера модели нет (см. Блок 4) - живой агент не запускается.")
else:
    try:
        import sys

        import cognopolis_mcp_server
        from mcp import StdioServerParameters
        from smolagents import OpenAIServerModel, ToolCallingAgent, ToolCollection

        async def stored_wood():
            async with Client(cognopolis_mcp_server.mcp) as c:
                char = (await c.call_tool("get_character", {})).data
            return (char.get("stored") or {}).get("wood", 0)

        wood_before = await stored_wood()
        print("stored.wood до прогона:", wood_before)
        print()

        params = StdioServerParameters(
            command=sys.executable,
            args=[str(SERVER_DIR / "cognopolis_mcp_server.py")],
            env={**os.environ, "COGNOPOLIS_TOKEN": COGNOPOLIS_TOKEN},   # токен - подпроцессу, не протоколу
        )
        with ToolCollection.from_mcp(params, trust_remote_code=True) as tc_live:
            agent = ToolCallingAgent(
                tools=[*tc_live.tools],
                model=OpenAIServerModel(model_id=LOCAL_MODEL, api_base=LOCAL_BASE, api_key="local"),
                max_steps=25,
                instructions=LIVE_INSTRUCTIONS,
            )
            agent.run("Добудь одно дерево (wood) и вернись домой на (0, 0), чтобы оно попало на склад.")

        calls = [t.name for step in agent.memory.steps
                 for t in (getattr(step, "tool_calls", None) or [])]
        wood_after = await stored_wood()
        print()
        print("Факт-чек по agent.memory.steps - цепочка вызовов:", calls)
        print(f"Факт-чек по миру: stored.wood {wood_before} -> {wood_after} "
              f"(дельта {wood_after - wood_before})")
        if wood_after > wood_before:
            print("Дерево на складе: агент добыл его через ваш MCP-сервер над живым API.")
        else:
            print("Дельта нулевая - загляните в лог агента: скорее всего, он не дошёл до дома (0, 0).")
    except Exception as e:
        print("Живой прогон не прошёл -> мягкий пропуск:", repr(e))

## Блок 6. Клиент №2, который вы не писали

Всё это время оба клиента — fastmcp и smolagents — были из вашего же ноутбука. Финальная проверка из лекции — подключить клиента, к которому вы не имеете отношения, и увидеть в нём те же инструменты. Это делается в терминале на вашей машине, не в ноутбуке.

**MCP Inspector** — отладочный клиент от авторов протокола (нужен Node.js; `npx` скачает пакет сам). CLI-режим печатает каталог прямо в терминал — запускайте из каталога модуля (папки с ноутбуком), путь к серверу — через `mcp-server/`:

```bash
npx -y @modelcontextprotocol/inspector --cli python mcp-server/woodcutter_server.py --method tools/list
```

Без `--cli` откроется веб-интерфейс, где те же tools и resources можно позвать кнопками. Обратите внимание: Inspector сам запустил ваш сервер подпроцессом (stdio из Блока 3) и сам сделал discovery — те же `tools/list` и `tools/call`, и никакой обвязки под него вы не писали.

**Claude Code** — то же самое одной командой:

```bash
claude mcp add woodcutter -- python mcp-server/woodcutter_server.py
```

Для живого сервера токен передаётся через `--env` — ровно так же, как мы передавали его подпроцессу в Блоке 5:

```bash
claude mcp add cognopolis --env COGNOPOLIS_TOKEN=<ваш-под-токен> -- python mcp-server/cognopolis_mcp_server.py
```

После этого команда `/mcp` в Claude Code покажет сервер и его инструменты, а просьба «сходи за деревом» превратится в те самые `tools/call`. Критерий из лекции выполнен, когда **одни и те же инструменты видны из двух разных клиентов**: smolagents (Блок 4) и Inspector или Claude Code — клиент, которого вы не писали. И discovery работает в обоих: уберите инструмент, как в Блоке 3, — он пропадёт из обоих клиентов без единой правки на их стороне.

Claude Desktop и Cursor подключаются так же, только через конфиг-файл — это Задача 4.

## Задачи

Задачи 1–3 работают прямо в ноутбуке: у каждой готовый рабочий образец (ноутбук остаётся зелёным на `Run all`), ваша работа — разобрать его, изменить под себя и прогнать снова. Живая часть Задачи 3 требует под-токена, но её главная — discovery — часть работает keyless. Задача 4 — мостик из ноутбука в ваш настоящий клиент.

Подход тот же, что в 11.5 и 13.6: запустите образец, убедитесь, что проверки сходятся, а потом ломайте и чините — меняйте схемы, убирайте и добавляйте примитивы и смотрите на сервер глазами клиента, через `tools/list` и `resources/list`.

### Задача 1. Второй resource: `woodcutter://stock`

У сервера лесоруба один resource — карта. Добавьте второй: склад `woodcutter://stock` — что уже сдано. Это чтение, мир оно не меняет — значит, примитив resource, не tool.

Образец ниже расширяет сервер без правки канонического файла: `woodcutter_server_stock.py` импортирует готовый объект `mcp` и довешивает на него один resource. `%%writefile` кладёт файл в `mcp-server/` — обязательно рядом с оригиналом: `from woodcutter_server import ...` ищет его в том же каталоге. В жизни вы бы просто дописали функцию в сам `woodcutter_server.py` — приём с импортом здесь только ради того, чтобы остальные ячейки видели файл нетронутым. Проверка — глазами клиента: `resources/list` теперь из двух URI.

In [ ]:
%%writefile mcp-server/woodcutter_server_stock.py
# Задача 1: тот же лесоруб + второй resource woodcutter://stock (склад).
# Старый сервер переиспользуем импортом; в жизни вы бы дописали функцию прямо в woodcutter_server.py.
from woodcutter_server import forest, mcp


@mcp.resource("woodcutter://stock")
def get_stock() -> dict:
    """Склад лесоруба: что уже сдано через deposit. Только чтение."""
    return {"stock": dict(forest.stock)}


if __name__ == "__main__":
    mcp.run()

In [ ]:
async with Client(str(SERVER_DIR / "woodcutter_server_stock.py")) as client:
    uris = sorted(str(r.uri) for r in await client.list_resources())
    stock = json.loads((await client.read_resource("woodcutter://stock"))[0].text)

print("resources/list:", uris)
print("woodcutter://stock:", stock)

assert uris == ["woodcutter://map", "woodcutter://stock"], "второй resource появился в каталоге"
assert stock == {"stock": {}}, "подпроцесс свежий - склад пока пуст"
print()
print("Дальше сами: в этом же клиенте сдайте пару wood через call_tool (move -> gather -> deposit)")
print("и перечитайте woodcutter://stock - склад в ресурсе оживёт.")

### Задача 2. Карта как tool: что теряется

Микропроверка из лекции, но наоборот: оформите карту **как tool** и посмотрите, что потерялось. Образец — мини-сервер `map_as_tool_server.py`: то же самое чтение, но под декоратором `@mcp.tool`.

Работает? Работает. Что потерялось — видно в discovery: `resources/list` пуст, чтение прикинулось действием. Модель теперь «вызывает действие» там, где нужно просто подтянуть контекст, — разделение `GET`/`POST`, за которое вы боролись в 11.5, стёрлось. А что приобрелось — тоже скажем честно: такой сервер увидит `ToolCollection.from_mcp`, который resources не умеет. Именно поэтому живой cognopolis-сервер держит карту в обоих примитивах сразу — осознанная плата за совместимость, а не небрежность.

«Вернуть как было» здесь ничего не нужно: канонический `woodcutter_server.py` мы не трогали — правильная карта-resource так и живёт в нём. Если вы экспериментировали в самом файле — верните `@mcp.resource("woodcutter://map")` на место и перепроверьте `resources/list`.

In [ ]:
%%writefile mcp-server/map_as_tool_server.py
# Задача 2: та же карта, но нарочно оформлена tool-ом, а не resource-ом.
from fastmcp import FastMCP

from woodcutter_server import forest

mcp = FastMCP("woodcutter-map-as-tool")


@mcp.tool
def get_map() -> dict:
    """Карта мок-леса: где лесоруб, узлы ресурсов и склад. То же чтение, что
    woodcutter://map, но клиент теперь видит его как действие."""
    return {"pos": list(forest.pos), "home": list(forest.home),
            "nodes": [{"pos": list(p), "resource": r} for p, r in forest.nodes.items()]}


if __name__ == "__main__":
    mcp.run()

In [ ]:
async with Client(str(SERVER_DIR / "map_as_tool_server.py")) as client:
    t_names = sorted(t.name for t in await client.list_tools())
    r_uris = [str(r.uri) for r in await client.list_resources()]
    map_via_tool = (await client.call_tool("get_map", {})).data

print("tools/list:    ", t_names)
print("resources/list:", r_uris or "пусто")
print("Карта через tools/call:", json.dumps(map_via_tool, ensure_ascii=False))

assert t_names == ["get_map"] and r_uris == []
print()
print("Данные те же, но примитив врёт: чтение выглядит действием, а resources/list пуст.")
print("Потерялось разделение GET/POST; приобрелась совместимость с клиентами, которые")
print("умеют только tools. Такой обмен делают осознанно - как в живом cognopolis-сервере.")

### Задача 3. Новый узкий tool для живого сервера

У живого API Cognopolis поверхностей больше, чем обернул наш сервер: лента событий `GET /events`, предпросмотр боя `POST /actions/fight/preview`, публичные справочники — полная карта в [ориентировке по игровому API](https://github.com/ITrubnikov/Train_of_Thought/blob/main/docs/game-api/index.mdx) и на kindomklaster.com/docs. Добавьте в сервер один **новый узкий tool** — по канону 11.5: одно имя, одна цель, честный контракт.

Образец — `get_events`: последние события поселения. Главная проверка — **discovery**: клиент увидел новый инструмент, хотя на его стороне не изменилось ничего. Эта часть работает даже без токена — `tools/list` не зовёт живой API. Живой вызов инструмента — только при токене.

In [ ]:
%%writefile mcp-server/cognopolis_mcp_server_v2.py
# Задача 3: тот же живой сервер + один новый узкий tool. Старое переиспользуем импортом;
# в жизни вы бы дописали функцию прямо в cognopolis_mcp_server.py.
from cognopolis_mcp_server import _request, mcp


@mcp.tool
def get_events(limit: int = 10) -> dict:
    """Последние события своего поселения (добыча, бои, постройки). Чтение, мир не меняет.

    Args:
        limit: сколько последних событий вернуть.
    """
    res = _request("GET", f"/events?limit={limit}", auth=True)
    if isinstance(res, dict):        # объясняющая ошибка {error: {code, message}} - отдаём как есть
        return res
    # живой API отдаёт массив событий, а tool обещает dict: заворачиваем в конверт
    # {events: [...]} - клиенты получают единую форму ответа (буква D чек-листа 11.5)
    return {"events": res}


if __name__ == "__main__":
    mcp.run()

In [ ]:
async with Client(str(SERVER_DIR / "cognopolis_mcp_server_v2.py")) as client:   # discovery без токена и сети
    tools_v2 = await client.list_tools()

names_v2 = sorted(t.name for t in tools_v2)
print("tools/list v2:", names_v2)
move_schema = next(t for t in tools_v2 if t.name == "move").inputSchema
print("enum направлений из Literal:",
      json.dumps(move_schema["properties"]["direction"], ensure_ascii=False))

assert "get_events" in names_v2, "новый tool виден клиенту без правок на его стороне"
assert "north" in json.dumps(move_schema)

if COGNOPOLIS_TOKEN:
    import cognopolis_mcp_server_v2
    async with Client(cognopolis_mcp_server_v2.mcp) as client:
        events = (await client.call_tool("get_events", {"limit": 3})).data
    print()
    print("Живой вызов get_events:", json.dumps(events, ensure_ascii=False)[:400])
else:
    print()
    print("Токена нет - живой вызов get_events пропущен; главное (discovery) уже показано.")

### Задача 4 (мостик в бой). Ваш сервер — в Claude Desktop или Claude Code

Шаг 4 домашки лекции: пропишите сервер в клиент, которым пользуетесь каждый день. Проверка этой задачи происходит не в ноутбуке, а у вас в клиенте; задача локальная — в Colab/Kaggle ячейка ниже напечатает пути облачной машины, перенесите файлы к себе.

- **Claude Code** — командой из Блока 6: `claude mcp add woodcutter -- python mcp-server/woodcutter_server.py` (из каталога модуля или с абсолютным путём). Проверка — `/mcp` показывает сервер, инструменты зовутся из чата.
- **Claude Desktop** — через конфиг `claude_desktop_config.json` (Settings → Developer → Edit Config): добавьте сервер в `mcpServers` с командой запуска — после перезапуска инструменты лесоруба появятся в интерфейсе. Cursor и другие MCP-клиенты настраиваются той же парой «команда + аргументы».

Ячейка ниже соберёт готовый фрагмент конфига с абсолютными путями — скопируйте его как есть. Для живого сервера не забудьте блок `env` с `COGNOPOLIS_TOKEN`: как и в Блоке 5, токен передаётся окружением подпроцесса, а не текстом в чате.

In [ ]:
import sys

desktop_config = {
    "mcpServers": {
        "woodcutter": {
            "command": sys.executable,
            "args": [str(SERVER_DIR / "woodcutter_server.py")],
        },
        "cognopolis": {
            "command": sys.executable,
            "args": [str(SERVER_DIR / "cognopolis_mcp_server.py")],
            "env": {"COGNOPOLIS_TOKEN": "<вставьте под-токен при копировании>"},
        },
    }
}
print("Фрагмент для claude_desktop_config.json (пути уже абсолютные):")
print(json.dumps(desktop_config, ensure_ascii=False, indent=2))
print()
print("Критерий из лекции: одни и те же инструменты видны из двух клиентов - из smolagents")
print("и из клиента, который вы не писали; уберёте инструмент - пропадёт в обоих.")

## Что дальше

Вы прошли всю троицу части V: tool (11.5) — одно действие с контрактом, skill (13.6) — процедура в памяти агента, MCP (14.5) — весь сервис, отданный наружу по стандарту. Ваш `woodcutter_server.py` — «контракт, вынесенный в сервис» в одном файле: три примитива, discovery, транспорт.

Дальше — Часть VI «Одиночные агенты по автономности»: из этих кирпичей собираются самостоятельные агенты. А `cognopolis_mcp_server.py` не выбрасывайте: свой MCP-сервер вокруг игрового API — прямая заготовка к капстоуну. Подключив его к Claude Code (Модуль 11) и к оркестратору LangGraph (Модуль 15), вы дадите готовому агенту доступ к миру одним конфигом, а не переписыванием обвязки. Расширяйте его по образцу Задачи 3: каждый новый узкий tool — ещё одно действие, которое разом получают все клиенты.

Готовый пример такого подключения уже лежит в репозитории — [`spaces/module-14-5-agent/`](https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/spaces/module-14-5-agent): локальный чат-агент на живой игре (Gradio + LM Studio), близнец зеркал 11.5 и 13.6 — с одной принципиальной разницей. В его файле нет ни одного определения игрового tool: инструменты приезжают из вашего `cognopolis_mcp_server.py` через `ToolCollection.from_mcp`. В 11.5 и 13.6 tools были кодом самого файла агента — теперь их отдаёт MCP-сервер, и файл агента про инструменты не знает ничего. Сравните `app.py` трёх зеркал — это и есть выигрыш модуля, видимый глазами.

**Критерии приёма (проверяете сами):**

- ноутбук прогнан целиком (`Run all`) keyless без ошибок;
- сервер поднимается по stdio, хотя бы одно чтение оформлено как resource, а не tool (Блоки 2–3);
- одни и те же инструменты видны из двух разных клиентов: smolagents (Блок 4) и клиент, который вы не писали, — Inspector, Claude Code или Claude Desktop (Блок 6, Задача 4);
- удаление инструмента отражается в `tools/list` без правок на стороне клиента (Блок 3);
- второй resource появился в `resources/list` (Задача 1); разница tool против resource объяснена на опыте (Задача 2); новый узкий tool живого сервера виден через discovery (Задача 3).

Артефакт для сдачи — публичная ссылка на прогнанный ноутбук, в чат курса как `[Модуль 14.5, ДЗ] {ссылка}`.

Если так — вы своими руками увидели, что такое «контракт, вынесенный в сервис», и домашка сдана, преподаватель не нужен.